In [40]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uproot as up
import pickle
import nue_booster
import importlib
importlib.reload(nue_booster)

import awkward

In [41]:
# SURPRISE SAMPLES
u_nu4a = up.open("/exp/uboone/data/users/kpletcher/slimmedFiles/slimmedMCC910/run4a/nuepresel/checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_4a.root")["nuselection"]["SubRun"]
u_nu4a_nufilter = up.open("/exp/uboone/data/users/kpletcher/slimmedFiles/slimmedMCC910/run4a/nuepresel/checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_4a.root")["nuselection"]["NeutrinoSelectionFilter"]

# u_nu4a = up.open("/pnfs/uboone/persistent/users/uboonepro/surprise/retuple/BNB/checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_4a.root")["nuselection"]["SubRun"]
# u_nu4a_nufilter = up.open("/pnfs/uboone/persistent/users/uboonepro/surprise/retuple/BNB/checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_4a.root")["nuselection"]["NeutrinoSelectionFilter"]

In [42]:
pd.set_option('display.max_rows', None)

vars = ["pot", "run", "subRun"]
vars_query = ["run", "sub", "nu_pdg", "ccnc", "mcf_pass_ncpi0"]

NUEQUERY = '(abs(nu_pdg)==12 & ccnc==0)'
NCPI0QUERY = '(mcf_pass_ncpi0==1)'
MCQUERY = '((abs(nu_pdg)==12 & ccnc==0) | mcf_pass_ncpi0==1)'

nu4a_subRun = u_nu4a.arrays(vars, library='pd')
nu4a_nueSelFilter = u_nu4a_nufilter.arrays(vars_query, library='pd')
# nu4a = pd.merge(nu4a_subRun, nu4a_nueSelFilter, left_on=["run","subRun"], right_on=["run","sub"], how="right")
nu4a = pd.merge(nu4a_subRun, nu4a_nueSelFilter, left_on=["run","subRun"], right_on=["run","sub"], how="left")

# print(nu4a_subRun)
# print(nu4a_nueSelFilter)
print(nu4a)

nu4a_ccnue = nu4a.query(NUEQUERY)
nu4a_ncpi0 = nu4a.query(NCPI0QUERY)
nu4a_no_ccnue_ncpi0 = nu4a.query('~'+MCQUERY)

train_nu4a, test_nu4a = train_test_split(nu4a_no_ccnue_ncpi0, test_size=0.5, random_state=1990)

# train_nue = pd.concat([train_nue_truth, nu_ccnue], ignore_index=True)
# train_ncpi0 = pd.concat([train_ncpi0_truth, nu_ncpi0], ignore_index=True)

                pot    run  subRun    sub  nu_pdg  ccnc  mcf_pass_ncpi0
0      1.296846e+16  18961       1    1.0    14.0   1.0             1.0
1      1.681176e+16  18961       4    NaN     NaN   NaN             NaN
2      2.243036e+16  18961       5    NaN     NaN   NaN             NaN
3      9.985432e+15  18961       8    NaN     NaN   NaN             NaN
4      1.087462e+16  18961       9    NaN     NaN   NaN             NaN
5      2.364614e+16  18961      10    NaN     NaN   NaN             NaN
6      6.420258e+15  18961      14    NaN     NaN   NaN             NaN
7      2.087777e+16  18961      17    NaN     NaN   NaN             NaN
8      1.726026e+16  18961      19    NaN     NaN   NaN             NaN
9      2.113662e+16  18961      20    NaN     NaN   NaN             NaN
10     1.058455e+16  18961      22    NaN     NaN   NaN             NaN
11     2.272361e+16  18961      25    NaN     NaN   NaN             NaN
12     1.033862e+16  18961      26    NaN     NaN   NaN         

In [43]:
# Run 4a Test POT
train_nu4a_pot = 0
test_nu4a_pot = 0
nu4a_ccnue_pot = 0
nu4a_ncpi0_pot = 0
nu4a_no_ccnue_ncpi0_pot = 0

for i, row in train_nu4a.iterrows():
    train_nu4a_pot+=row["pot"]

for i, row in test_nu4a.iterrows():
    test_nu4a_pot+=row["pot"]

for i, row in nu4a_ccnue.iterrows():
    nu4a_ccnue_pot+=row["pot"]

for i, row in nu4a_ncpi0.iterrows():
    nu4a_ncpi0_pot+=row["pot"]

for i, row in nu4a_no_ccnue_ncpi0.iterrows():
    nu4a_no_ccnue_ncpi0_pot+=row["pot"]

In [44]:
print("Nu 4a Train Sample =", train_nu4a_pot)
print("Nu 4a Test Sample =", test_nu4a_pot)
print("Nu 4a CC Nue Events =", nu4a_ccnue_pot)
print("Nu 4a NC Pi0 Events =", nu4a_ncpi0_pot)
print("Nu 4a Non CC Nue and NC Pi0 Events =", nu4a_no_ccnue_ncpi0_pot)

Nu 4a Train Sample = 1.1226691986764752e+20
Nu 4a Test Sample = 1.1290087114499372e+20
Nu 4a CC Nue Events = 2.2160777463191306e+18
Nu 4a NC Pi0 Events = 9.568991486037459e+18
Nu 4a Non CC Nue and NC Pi0 Events = 2.2516779101264124e+20


In [46]:
print("Total Run 4a POT, No CC Nue or NC Pi0 =", train_nu4a_pot+test_nu4a_pot)
print("Total Run 4a Nu Overlay POT =", nu4a_ccnue_pot+nu4a_ncpi0_pot+nu4a_no_ccnue_ncpi0_pot)

Total Run 4a POT, No CC Nue or NC Pi0 = 2.2516779101264124e+20
Total Run 4a Nu Overlay POT = 2.3695286024499783e+20
